# Multi-Agent Debate System — Bias Analysis

Loads `results.jsonl` produced by `run_experiment.py` and checks whether judge scores are swayed by:
- the presentation **order** of the two arguments (swapping which side is shown first), and
- the **length** of the arguments — each side writes its case to several word budgets (e.g. 60 / 120 / 180), judged as same-length pairs, and we look for scores climbing with length,

and whether a **heterogeneous** judge panel (3 different models) shows less of this bias than a **homogeneous** one (1 model x3).

Every metric below comes from `debate_system/analysis.py`, so these numbers match the Streamlit dashboard exactly.

In [ ]:
from pathlib import Path

from debate_system import analysis

df = analysis.load_results(Path("results/run.jsonl"))
df.head()

## 1. Order effect

If judges were unbiased, whichever argument is shown first ("Debater 1") should score the same on average as the one shown second, and the declared winner shouldn't flip just because we swap which side is first.

In [ ]:
oe = analysis.order_effect(df)
print(f"Average (first - second) total score: {oe['first_minus_second']:+.2f}")
print("(0 = no order bias; positive = judges favor whichever argument is shown first)")
oe["by_order"]

In [ ]:
print(f"Winner flip rate on order swap: {analysis.flip_rate(df):.1%}")
analysis.winner_flips(df).head()

## 2. Length effect

Each pair is length-matched (both sides written to the same word budget), so verbosity bias can't show up as a PRO/CON gap — it shows up as the **overall score level rising** as the arguments get longer. `score_climb` is the mean score at the longest target minus the shortest; `corr_length_score` correlates length with score; `mean_target_miss` is how far the models landed from the budget they were asked to hit.

In [ ]:
le = analysis.length_effect(df)
print(f"corr(length, score):              {le['corr_length_score']:+.2f}")
print(f"score climb (longest - shortest): {le['score_climb']:+.2f}")
print(f"mean words off target:            {le['mean_target_miss']:.1f}")
le["by_length"]

In [ ]:
import matplotlib.pyplot as plt

by_length = le["by_length"]
plt.figure(figsize=(5, 4))
plt.bar([str(x) for x in by_length.index], by_length.values)
plt.title("Mean score vs argument length (a rising bar = length bias)")
plt.xlabel("target words")
plt.ylabel("mean score (both debaters)")
plt.tight_layout()
plt.show()

## 3. Does a diverse judge panel reduce bias?

Compares the order-flip rate and the length bias (score climb with length) between the `homogeneous` (1 model x3) and `heterogeneous` (3 different models) panels.

In [ ]:
analysis.panel_comparison(df)

## 4. Summary

The headline numbers used in the writeup/slides.

In [ ]:
import json

print(json.dumps(analysis.summary(df), indent=2, default=float))